In [ ]:
!pip install -q requests beautifulsoup4 cloudscraper ipywidgets

# romsfun — PSP ROM Downloader

Scrapes romsfun.com game pages and downloads the ISO/CSO files directly to Google Drive.

---

## Source

- **Site:** https://romsfun.com
- **Section:** PlayStation Portable (PSP)
- **How to find URLs:** browse to any game page (e.g. `romsfun.com/roms/playstation-portable/<game>.html`) and paste the page URL — not the download link, the game page itself.

## What it does

1. Takes a list of romsfun game page URLs from the input widget.
2. Scrapes each page to find the direct download link (`.iso`, `.zip`, `.7z`, `.rar`, `.cso`, `.pbp`).
3. Mounts Google Drive and downloads each file **directly** to `MyDrive/CollabMedia/downloads/romsfun/`.
4. Supports resume — interrupted downloads pick up where they left off.

## How to use

1. Run **Cell 1** (`pip install`) — installs dependencies.
2. Run **Cell 2** — shows the URL input box.
3. Paste your romsfun game page URLs in the box, one per line.
4. Run **Cell 3** — mounts Drive, reads URLs, sets up scraper.
5. Run **Cell 4** — defines download functions (no output).
6. Run **Cell 5** — starts downloading directly to Drive. Shows progress per file.

## Known limitations

- Some pages serve the download link via JavaScript — the scraper won't find it and will print `WARN: no download URL found`. Check those pages manually.
- If you get a 403, the site may have detected the bot. Try re-running with a longer `time.sleep` delay.
- Drive writes are slower than local disk — expected for large ISOs.
- Colab disconnects after ~12 h of inactivity — but files already on Drive are safe.

## Output

```
MyDrive/
└── CollabMedia/
    └── downloads/
        └── romsfun/
            ├── game1.iso
            ├── game2.zip
            └── ...
```

In [ ]:
import ipywidgets as widgets
from IPython.display import display

url_box = widgets.Textarea(
    value="",
    placeholder="Paste romsfun.com page URLs here, one per line",
    layout=widgets.Layout(width="100%", height="260px"),
)
display(widgets.Label("romsfun page URLs (one per line):"), url_box)


In [ ]:
import cloudscraper
from bs4 import BeautifulSoup
import os, time, re
from urllib.parse import urljoin
from google.colab import drive

drive.mount("/content/drive")

PAGES = [u.strip() for u in url_box.value.splitlines() if u.strip()]
print(f"{len(PAGES)} URLs queued.")

OUT_DIR = "/content/drive/MyDrive/CollabMedia/downloads/romsfun"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output → {OUT_DIR}")

scraper = cloudscraper.create_scraper()
scraper.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
})

In [ ]:
def find_download_url(page_url, version_num="1"):
    resp = scraper.get(page_url, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")

    # 1. Direct file link
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if re.search(r"\.(iso|zip|7z|rar|cso|pbp)(\?|$)", href, re.I):
            return urljoin(page_url, href), None, None

    # 2. Locate download hub
    hub_url = None
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/download/" in href and not href.endswith(("#", ".png", ".jpg")):
            hub_url = urljoin(page_url, href)
            break

    if not hub_url:
        if "/download/" in page_url:
            hub_url = page_url
        else:
            return None, None, None

    # 3. Locate version subpage
    version_url = hub_url
    if not re.search(r"/download/[^/]+/\d+$", hub_url):
        hub_resp = scraper.get(hub_url, timeout=30)
        hub_soup = BeautifulSoup(hub_resp.text, "html.parser")
        v_links = [urljoin(hub_url, a["href"]) for a in hub_soup.find_all("a", href=True) if re.search(r"/download/[^/]+/\d+$", a["href"])]
        if v_links:
            version_url = v_links[0]
        else:
            version_url = hub_url.rstrip("/") + "/" + version_num

    # 4. Request dynamic download token via AJAX
    ajax_url = "https://romsfun.com/wp-admin/admin-ajax.php"
    headers = {
        "Referer": version_url,
        "X-Requested-With": "XMLHttpRequest",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8"
    }
    ajax_resp = scraper.post(ajax_url, data="action=k_get_download", headers=headers, timeout=30)
    if ajax_resp.status_code == 200:
        try:
            data = ajax_resp.json()
            if data.get("success") and data.get("data", {}).get("download_url"):
                dl_data = data["data"]
                return dl_data["download_url"], dl_data.get("file_name"), version_url
        except Exception:
            pass

    return None, None, None


def download_file(url, out_dir, file_name=None, version_url=None):
    clean_path = url.split("?")[0]
    ext = os.path.splitext(clean_path)[1] or ".zip"
    if file_name:
        safe_name = re.sub(r"[\\/*?:\"<>|]", "", file_name) + ext
    else:
        safe_name = url.split("/")[-1].split("?")[0]
    out_path = os.path.join(out_dir, safe_name)

    downloaded = 0
    if os.path.exists(out_path):
        downloaded = os.path.getsize(out_path)
        if downloaded > 0:
            print(f"  Resuming {safe_name} from {downloaded/1024/1024:.1f} MB...")
    else:
        print(f"  Downloading: {safe_name}")

    current_url = url
    total_bytes = 0
    max_retries = 30
    retries = 0
    start_time = time.time()

    while True:
        try:
            headers = {"Referer": "https://romsfun.com/"}
            if downloaded > 0:
                headers["Range"] = f"bytes={downloaded}-"

            with scraper.get(current_url, stream=True, timeout=60, headers=headers) as r:
                if r.status_code in (403, 410) and version_url:
                    print("\n  [Token expired, refreshing...]")
                    new_url, _, _ = find_download_url(version_url)
                    if new_url:
                        current_url = new_url
                        continue
                if r.status_code not in (200, 206):
                    r.raise_for_status()
                if downloaded == 0:
                    total_bytes = int(r.headers.get("content-length", 0))
                elif total_bytes == 0 and "content-range" in r.headers:
                    total_bytes = int(r.headers["content-range"].split("/")[-1])

                if total_bytes > 0 and downloaded >= total_bytes:
                    print(f"  Already complete ({downloaded/1024/1024:.1f} MB).")
                    break

                with open(out_path, "ab" if downloaded > 0 else "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 512):
                        if chunk:
                            f.write(chunk)
                            downloaded += len(chunk)
                            elapsed = time.time() - start_time
                            speed_mb = (downloaded / (1024 * 1024)) / (elapsed if elapsed > 0 else 1)
                            if total_bytes > 0:
                                pct = downloaded / total_bytes * 100
                                print(f"\r  {pct:.1f}% ({downloaded/1024/1024:.1f}/{total_bytes/1024/1024:.1f} MB) - {speed_mb:.2f} MB/s", end="")

            if total_bytes > 0 and downloaded >= total_bytes:
                print(f"\n  Done: {out_path}")
                break
        except Exception as err:
            retries += 1
            if retries > max_retries:
                raise
            print(f"\n  [Dropped: {err}. Resuming from {downloaded/1024/1024:.1f} MB...]")
            time.sleep(3)
            if version_url:
                try:
                    new_url, _, _ = find_download_url(version_url)
                    if new_url:
                        current_url = new_url
                except Exception:
                    pass


In [ ]:
failed = []

for page in PAGES:
    name = page.split("/")[-1].replace(".html", "")
    print(f"\n[{name}]")
    try:
        dl_url, file_name, version_url = find_download_url(page)
        if not dl_url:
            print("  WARN: no download URL found — check page manually")
            print(f"  Page: {page}")
            failed.append(page)
            continue
        print(f"  URL: {dl_url[:80]}...")
        download_file(dl_url, OUT_DIR, file_name, version_url)
    except Exception as e:
        print(f"  ERROR: {e}")
        failed.append(page)
    time.sleep(2)

print("\n=== DONE ===")
if failed:
    print("Failed (check manually):")
    for p in failed:
        print(" ", p)
else:
    print("All succeeded.")